# Study Case: Ecommerce Customer Dataset Normalization

## Normalization Planning

For this step we can use all kinds of tools based on preference to plan (or maybe visualize) the normalization process.

### UNF

<img src="unf.png" width="300">

### 1NF

<img src="1nf.png" width="300">

Notes:
- **PK**: `Customer ID` & `Purchase Date` because this pair is unique and can be used to identify each rows.

### 2NF

<img src="2nf.png" width="500">

Notes:
- Because `Costumer ID` has **functional dependency** with `Customer Age` and `Customer Name`, while `Purchase Date` doesn't. So this **customers** info must be separated into a single table with `Customer ID` as its PK.
- Then the `Customer ID` in the **main_table** now act as a FK that related to **customers** table.
- But after the we create the **customers** table, now the PK for **main_table**, which is `Purchase Date` is not entirely unique (you can see it from the dataset), so it can't be the main PK for this table and this makes **main_table** doesn't have a PK. However, there are 2 options for this case:
    1. We can ignore the PK in **main_table** for a while and continue the normalization process until we can call it normal. Then, later on when we are importing the csv dataset into the database, we can add an `ID` column with **serial** type as PK in the SQL script.
    2. Or, we add the `ID` column (integer) in **main_table** schema as PK first. Then, we hardcoded the data or by using scripts to fill this `ID` column in the existing dataset. And finally we can import it into database.

### 3NF

<img src="3nf.png" width="500">

Notes: 
- Because `ID` has **transitive dependency** with `Product Category ID` and `Product Category`, so must separate them into its own table called **product_categories** while keeping the relationship between `Product Category ID` (FK) in **main_table** and `Product Category ID` (PK) in **product_categories**.

### Additional

<img src="additional.png" width="700">

Notes:
- To improve the Normalization results, we can also create a separate table for `Gender` and `Payment Method` like this. This will eliminate data redundancy and ensure data integrity by establishing a "single source of truth" for specific, reusable data elements. 

## Code Implementation

Because the dataset is in `csv` format, we can use **pandas** (python) to do the normalization process by separate the dataframe into several ones by following the schema plan that we made before.

### UNF

In [13]:
import pandas as pd

df = pd.read_csv("ecommerce_customer_data.csv")
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Gender,Churn,Product Category ID
0,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,37,0.0,Christine Hernandez,Male,0,1
1,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,37,0.0,Christine Hernandez,Male,0,2
2,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,37,0.0,Christine Hernandez,Male,0,2
3,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,37,0.0,Christine Hernandez,Male,0,3
4,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,49,0.0,James Grant,Female,1,2


### 1NF

In [14]:
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Gender,Churn,Product Category ID
0,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,37,0.0,Christine Hernandez,Male,0,1
1,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,37,0.0,Christine Hernandez,Male,0,2
2,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,37,0.0,Christine Hernandez,Male,0,2
3,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,37,0.0,Christine Hernandez,Male,0,3
4,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,49,0.0,James Grant,Female,1,2


Nothing's changed because all data is already atomic and we know that `Customer ID` and `Purchase Date` in this table are the **PK**.

### 2NF

#### create customer table

In [15]:
customer = df.copy()
customer = customer[["Customer ID", "Customer Age", "Customer Name"]].drop_duplicates(ignore_index=True)

customer

,Customer ID,Customer Age,Customer Name
0,46251,37,Christine Hernandez
1,13593,49,James Grant
2,28805,19,Jose Collier
3,28961,55,James Stein
4,12163,67,Sonia Moreno
...,...,...,...
49668,33308,55,Michelle Flores
49669,48835,42,Jeremy Rush
49670,21019,41,Tina Craig
49671,49234,34,Jennifer Cooper


#### modify the main table

In [16]:
newDf = df.drop(columns=["Customer Age", "Customer Name"])

newDf["ID"] = newDf.index + 1

# re-arrange the ID column into first position (index 0)
newDf.insert(0, 'ID', newDf.pop("ID"))

newDf.head()

,ID,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Returns,Gender,Churn,Product Category ID
0,1,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,0.0,Male,0,1
1,2,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,0.0,Male,0,2
2,3,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,0.0,Male,0,2
3,4,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,0.0,Male,0,3
4,5,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,0.0,Female,1,2


From the planning notes that we have, we choose the 2nd option. But if we were to choose the first option it should be like this.

In [17]:
newDf_first = df.drop(columns=["Customer Age", "Customer Name"])
newDf_first.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Returns,Gender,Churn,Product Category ID
0,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,0.0,Male,0,1
1,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,0.0,Male,0,2
2,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,0.0,Male,0,2
3,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,0.0,Male,0,3
4,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,0.0,Female,1,2


It's still like nothing change at all, but when we are importing it to database, we must add `ID` column in the SQL like this.
```postgresql
create table main_table (
    "ID" serial primary key,
    ....
)
```

but for now, we just replace the existing dataframe.

In [18]:
df = newDf

df.head()

,ID,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Returns,Gender,Churn,Product Category ID
0,1,46251,2020-09-08 09:38:32,Electronics,12,3,740,Credit Card,0.0,Male,0,1
1,2,46251,2022-03-05 12:56:35,Home,468,4,2739,PayPal,0.0,Male,0,2
2,3,46251,2022-05-23 18:18:01,Home,288,2,3196,PayPal,0.0,Male,0,2
3,4,46251,2020-11-12 13:13:29,Clothing,196,1,3509,PayPal,0.0,Male,0,3
4,5,13593,2020-11-27 17:55:11,Home,449,1,3452,Credit Card,0.0,Female,1,2


### 3NF

#### create product categories

In [19]:
product_category = df.copy()
product_category = product_category[["Product Category ID", "Product Category"]].drop_duplicates(ignore_index=True)

product_category.head()

,Product Category ID,Product Category
0,1,Electronics
1,2,Home
2,3,Clothing
3,4,Books


In [20]:
# modify main table again
newDf = df.drop(columns="Product Category")

# replace existing dataframe
df = newDf

### Normalization Result 

In [21]:
customer.head()

,Customer ID,Customer Age,Customer Name
0,46251,37,Christine Hernandez
1,13593,49,James Grant
2,28805,19,Jose Collier
3,28961,55,James Stein
4,12163,67,Sonia Moreno


In [22]:
product_category.head()

,Product Category ID,Product Category
0,1,Electronics
1,2,Home
2,3,Clothing
3,4,Books


In [23]:
df.head()

,ID,Customer ID,Purchase Date,Product Price,Quantity,Total Purchase Amount,Payment Method,Returns,Gender,Churn,Product Category ID
0,1,46251,2020-09-08 09:38:32,12,3,740,Credit Card,0.0,Male,0,1
1,2,46251,2022-03-05 12:56:35,468,4,2739,PayPal,0.0,Male,0,2
2,3,46251,2022-05-23 18:18:01,288,2,3196,PayPal,0.0,Male,0,2
3,4,46251,2020-11-12 13:13:29,196,1,3509,PayPal,0.0,Male,0,3
4,5,13593,2020-11-27 17:55:11,449,1,3452,Credit Card,0.0,Female,1,2


#### Export to csv

In [24]:
customer.to_csv("results/customer.csv", index=False)
product_category.to_csv("results/product_category.csv", index=False)
df.to_csv("results/main_table.csv", index=False)

### Additional

#### Gender table

In [25]:
gender = df[["Gender"]].drop_duplicates(ignore_index=True)
gender.to_csv("results/gender.csv", index=False)

gender.head()

,Gender
0,Male
1,Female


#### Payment Method table

In [26]:
payment_method = df[["Payment Method"]].drop_duplicates(ignore_index=True)
payment_method.to_csv("results/payment_method.csv", index=False)

payment_method.head()

,Payment Method
0,Credit Card
1,PayPal
2,Cash
3,Crypto
